# YouTube + Vimeo QoE Report

Per-tier rendering of the concurrent YouTube + Vimeo experiment (`run_direct_youtube_vimeo_experiment.py`), 30 seconds of synchronized playback per bandwidth tier with 100 ms latency and pfifo. Set `BANDWIDTH_MBPS` (3, 6, or 10) before running.

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import display

sys.path.insert(0, str(Path("experiments").resolve()))
from youtube_vimeo_report_helpers import load_report_context

BANDWIDTH_MBPS = int(os.environ.get("BANDWIDTH_MBPS", "3"))
globals().update(load_report_context(BANDWIDTH_MBPS))
display(summary.round(3))


## Browser screenshots

Final rendered browser frames captured before each Selenium session closed.

In [ ]:
from IPython.display import Image

for app, label in (("youtube", "YouTube"), ("vimeo", "Vimeo")):
    screenshot = RESULT_DIR / f"{app}_screenshot.png"
    if screenshot.is_file():
        print(label)
        display(Image(filename=str(screenshot), width=900))
    else:
        print(f"{label}: screenshot not available")


## Playback screenshot timeline

Two rows show YouTube and Vimeo at five-second intervals during the measurement window.

In [ ]:
screenshot_times = list(range(0, 30, 5))
fig, axes = plt.subplots(2, len(screenshot_times), figsize=(18, 7), dpi=120)
for row_index, (app, label) in enumerate((("youtube", "YouTube"), ("vimeo", "Vimeo"))):
    for column_index, second in enumerate(screenshot_times):
        ax = axes[row_index, column_index]
        screenshot = RESULT_DIR / f'{app}_screenshots/{second:03d}s.png'
        if screenshot.is_file():
            ax.imshow(plt.imread(screenshot))
        else:
            ax.text(.5, .5, 'not captured', ha='center', va='center')
        ax.set_title(f'{label} — {second}s', fontsize=9)
        ax.axis('off')
plt.tight_layout()
plt.show()


## Per-application throughput

Traffic is attributed from observed TLS/QUIC server names (SNI). YouTube: `youtube.com`, `googlevideo.com`, `ytimg.com`, and `ggpht.com`. Vimeo: `vimeo.com` and `vimeocdn.com`. Broad Google IP ranges are deliberately not treated as YouTube because Chrome background services use the same ranges. Any remaining bytes are shown separately as "Unclassified".

In [ ]:
PCAP = next(RESULT_DIR.glob('*.pcap'))
CLIENT_IP = '172.16.1.1'
if all_qoe:
    window_start = min(row['timestamp'] for row in all_qoe)
    window_end = max(row['timestamp'] for row in all_qoe)
else:
    capture_bounds = subprocess.run(
        ['tshark', '-r', str(PCAP), '-T', 'fields', '-e', 'frame.time_epoch'],
        check=True, capture_output=True, text=True,
    ).stdout.splitlines()
    capture_times = [float(value) for value in capture_bounds if value]
    window_start, window_end = min(capture_times), max(capture_times)
window_seconds = window_end - window_start

sni_output = subprocess.run(
    ['tshark', '-r', str(PCAP), '-Y', 'tls.handshake.extensions_server_name',
     '-T', 'fields', '-e', 'ip.dst', '-e', 'tls.handshake.extensions_server_name'],
    check=True, capture_output=True, text=True,
).stdout
hosts_by_ip = {}
for line in sni_output.splitlines():
    fields = line.split('\t')
    if len(fields) >= 2 and fields[0] and fields[1]:
        hosts_by_ip.setdefault(fields[0], set()).update(h.lower() for h in fields[1].split(','))

youtube_markers = ('youtube.com', 'googlevideo.com', 'ytimg.com', 'ggpht.com')
vimeo_markers = ('vimeo.com', 'vimeocdn.com')
youtube_ips = {ip for ip, hosts in hosts_by_ip.items() if any(m in h for h in hosts for m in youtube_markers)}
vimeo_ips = {ip for ip, hosts in hosts_by_ip.items() if any(m in h for h in hosts for m in vimeo_markers)} - youtube_ips

def classify_ip(ip):
    if ip in youtube_ips:
        return 'YouTube'
    if ip in vimeo_ips:
        return 'Vimeo'
    return 'Unclassified'

def classify_packets(direction_filter, remote_ip_field):
    output = subprocess.run(
        ['tshark', '-r', str(PCAP), '-Y', direction_filter, '-T', 'fields',
         '-e', 'frame.time_epoch', '-e', remote_ip_field, '-e', 'frame.len'],
        check=True, capture_output=True, text=True,
    ).stdout
    rows = []
    for line in output.splitlines():
        fields = line.split('\t')
        if len(fields) < 3 or not fields[0] or not fields[1] or not fields[2]:
            continue
        timestamp = float(fields[0])
        if not window_start <= timestamp <= window_end:
            continue
        remote_ip = fields[1].split(',')[0]
        frame_bytes = int(fields[2].split(',')[0])
        rows.append((timestamp, remote_ip, frame_bytes, classify_ip(remote_ip)))
    frame = pd.DataFrame(rows, columns=['timestamp', 'remote_ip', 'frame_bytes', 'application'])
    frame['second'] = (frame.timestamp - window_start).astype(int)
    return frame

download_packets = classify_packets(f'ip.dst == {CLIENT_IP}', 'ip.src')
upload_packets = classify_packets(f'ip.src == {CLIENT_IP}', 'ip.dst')

applications = ['YouTube', 'Vimeo', 'Unclassified']
app_colors = {'YouTube': YOUTUBE_COLOR, 'Vimeo': VIMEO_COLOR, 'Unclassified': '#999999'}
bin_count = max(1, math.ceil(window_seconds))

def per_second_mbps(packets):
    app_bytes = (packets[packets.application.isin(applications)]
                 .groupby(['second', 'application']).frame_bytes.sum()
                 .unstack(fill_value=0)
                 .reindex(range(bin_count), fill_value=0)
                 .reindex(columns=applications, fill_value=0))
    return app_bytes * 8 / 1_000_000

download_mbps = per_second_mbps(download_packets)
upload_mbps = per_second_mbps(upload_packets)

def summarize(packets, app_mbps, label):
    rows = []
    for app in applications:
        total_bytes = packets.loc[packets.application == app, 'frame_bytes'].sum()
        rows.append({
            'application': app,
            f'{label} (MiB)': total_bytes / 2**20,
            'average during QoE window (Mbps)': total_bytes * 8 / window_seconds / 1_000_000,
            'peak 1-second bin (Mbps)': app_mbps[app].max(),
        })
    display(pd.DataFrame(rows).set_index('application').round(3))

print('Download:')
summarize(download_packets, download_mbps, 'downloaded')
print('Upload:')
summarize(upload_packets, upload_mbps, 'uploaded')


## Download throughput

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for app in applications:
    ax.plot(download_mbps.index, download_mbps[app], marker='o', color=app_colors[app], label=app, alpha=(1.0 if app != 'Unclassified' else 0.6))
ax.axhline(BANDWIDTH_MBPS, color='#333333', linestyle='--', alpha=.7, label=f'Configured bottleneck ({BANDWIDTH_MBPS} Mbps)')
ax.set(title=f'Per-application Download Throughput — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Downloaded Mbit in each 1-second bin')
ax.legend()
plt.tight_layout()
plt.show()


## Download throughput by remote IP

Top endpoints are selected by total downloaded bytes in this capture window. Labels use the observed TLS/QUIC hostname when available.

In [ ]:
endpoint_totals = download_packets.groupby('remote_ip').frame_bytes.sum().sort_values(ascending=False)
top_endpoint_ips = endpoint_totals.head(10).index.tolist()
endpoint_bytes = (download_packets[download_packets.remote_ip.isin(top_endpoint_ips)]
                  .groupby(['second', 'remote_ip']).frame_bytes.sum()
                  .unstack(fill_value=0)
                  .reindex(range(bin_count), fill_value=0)
                  .reindex(columns=top_endpoint_ips, fill_value=0))
endpoint_mbps = endpoint_bytes * 8 / 1_000_000
def endpoint_label(ip):
    names = sorted(hosts_by_ip.get(ip, set()))
    return f'{ip} — {names[0]}' if names else ip
endpoint_summary = pd.DataFrame({
    'remote IP': top_endpoint_ips,
    'observed hostname': [', '.join(sorted(hosts_by_ip.get(ip, set()))) or 'not observed' for ip in top_endpoint_ips],
    'downloaded MiB': [endpoint_totals[ip] / 2**20 for ip in top_endpoint_ips],
}).set_index('remote IP')
display(endpoint_summary.round(3))
fig, ax = plt.subplots(figsize=(12, 6), dpi=120)
for ip in top_endpoint_ips:
    ax.plot(endpoint_mbps.index, endpoint_mbps[ip], marker='o', markersize=3, label=endpoint_label(ip))
ax.axhline(BANDWIDTH_MBPS, color='#333333', linestyle='--', alpha=.7, label=f'Configured bottleneck ({BANDWIDTH_MBPS} Mbps)')
ax.set(title=f'Download Throughput by Remote IP — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from capture/QoE window start', ylabel='Downloaded Mbit in each 1-second bin')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()


## Upload throughput

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for app in applications:
    ax.plot(upload_mbps.index, upload_mbps[app], marker='o', color=app_colors[app], label=app, alpha=(1.0 if app != 'Unclassified' else 0.6))
ax.set(title=f'Per-application Upload Throughput — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Uploaded Mbit in each 1-second bin')
ax.legend()
plt.tight_layout()
plt.show()


## Buffer health

In [ ]:
youtube_buffer = [row['stats'].get('buffer_ahead_secs', 0) or 0 for row in youtube_qoe]
vimeo_buffer = [row['stats'].get('buffer_ahead_secs', 0) or 0 for row in vimeo_qoe]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.plot(youtube_seconds, youtube_buffer, marker='o', color=YOUTUBE_COLOR, label='YouTube')
ax.plot(vimeo_seconds, vimeo_buffer, marker='o', color=VIMEO_COLOR, label='Vimeo')
ax.axhline(0, color='#ff7f7f', linestyle='--', alpha=.7)
ax.set(title=f'Buffer Health — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Buffer ahead of playhead (seconds)')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Dropped frame rate

In [ ]:
def drop_percent(qoe):
    out = []
    for row in qoe:
        dropped = row['stats'].get('dropped_video_frames', 0) or 0
        total = row['stats'].get('total_video_frames', 0) or 0
        out.append(100 * dropped / max(1, total))
    return out

youtube_drop_pct = drop_percent(youtube_qoe)
vimeo_drop_pct = drop_percent(vimeo_qoe)
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.plot(youtube_seconds, youtube_drop_pct, marker='o', color=YOUTUBE_COLOR, label='YouTube')
ax.plot(vimeo_seconds, vimeo_drop_pct, marker='o', color=VIMEO_COLOR, label='Vimeo')
ax.set(title=f'Cumulative Dropped Frame Rate — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Dropped frames (% of decoded frames so far)')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Video resolution over time

In [ ]:
youtube_height = [row['stats'].get('video_height', 0) or 0 for row in youtube_qoe]
youtube_width = [row['stats'].get('video_width', 0) or 0 for row in youtube_qoe]
vimeo_height = [row['stats'].get('video_height', 0) or 0 for row in vimeo_qoe]
vimeo_width = [row['stats'].get('video_width', 0) or 0 for row in vimeo_qoe]

fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.step(youtube_seconds, youtube_height, where='post', color=YOUTUBE_COLOR, linewidth=2, label='YouTube')
ax.scatter(youtube_seconds, youtube_height, color=YOUTUBE_COLOR, s=24)
ax.step(vimeo_seconds, vimeo_height, where='post', color=VIMEO_COLOR, linewidth=2, label='Vimeo')
ax.scatter(vimeo_seconds, vimeo_height, color=VIMEO_COLOR, s=24)

resolution_labels = {}
for width, height in zip(youtube_width + vimeo_width, youtube_height + vimeo_height):
    if width and height:
        resolution_labels.setdefault(height, set()).add(f'{width}x{height}')
tick_heights = sorted(resolution_labels)
if tick_heights:
    ax.set_yticks(tick_heights, [' / '.join(sorted(resolution_labels[h])) for h in tick_heights])
ax.set(title=f'Video Resolution Over Time — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Rendered resolution')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Total video frames

In [ ]:
youtube_frames = [row['stats'].get('total_video_frames', 0) or 0 for row in youtube_qoe]
vimeo_frames = [row['stats'].get('total_video_frames', 0) or 0 for row in vimeo_qoe]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.plot(youtube_seconds, youtube_frames, marker='o', color=YOUTUBE_COLOR, label='YouTube')
ax.plot(vimeo_seconds, vimeo_frames, marker='o', color=VIMEO_COLOR, label='Vimeo')
ax.set(title=f'Total Video Frames — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Cumulative total video frames')
ax.legend(loc='best')
plt.tight_layout()
plt.show()
